# Save a conversation and continue it

Responses keep reasoning, calls, and assistant replies as separate entries in `messages`. Ordinary save/load preserves the available history. This notebook uses an offline SDK fixture so we can inspect exactly what the next request sends.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import httpx
from openai import AsyncOpenAI
from chatsnack import Chat

requests = []

def offline_response(request):
    """Return a small Responses conversation without a paid API call."""
    body = json.loads(request.content)
    requests.append(body)
    number = len(requests)
    output = [
        {"type": "reasoning", "id": f"rs_{number}", "summary": [],
         "encrypted_content": f"offline-reasoning-{number}"},
        {"type": "message", "id": f"msg_{number}", "role": "assistant",
         "phase": "final_answer", "status": "completed",
         "content": [{"type": "output_text", "text": "There are 12 snack boxes.", "annotations": []}]},
    ]
    return httpx.Response(200, json={
        "id": f"resp_{number}", "object": "response", "created_at": 1,
        "status": "completed", "model": body["model"], "output": output,
    })

sdk = AsyncOpenAI(api_key="offline", base_url="https://offline.invalid/v1",
                  http_client=httpx.AsyncClient(transport=httpx.MockTransport(offline_response)))
workspace = TemporaryDirectory()
folder = Path(workspace.name)


## Plain dialogue stays small

Authored replies stay scalar. Recorded replies add their item ID and phase; empty metadata and completed-status defaults stay out of the YAML.


In [ ]:
print(Chat().user("Hello.").asst("Blah").yaml)


## Author a reusable prompt

The source keeps `{sku}` until the call.


In [ ]:
template = folder / "StockHelper.yml"
template.write_text("""messages:
  - system: Answer stock questions briefly.
  - user: How many {sku} are available?
params:
  runtime: responses
  model: test-model
""", encoding="utf-8")
helper = Chat()
helper.load(str(template))
helper.ai.aclient = sdk  # Offline fixture; a configured live Chat needs no override.
thread = await helper.chat_a(sku="snack boxes")
print(thread.response)
print(thread.yaml)


## Save, reload, and continue

No `export_state` flag is needed. Utensil conversations use the same flow; pass the existing Python utensils when restoring a Chat that needs to execute them.


In [ ]:
saved = folder / "StockConversation.yml"
thread.save(str(saved))
restored = Chat()
restored.load(str(saved))
restored.ai.aclient = sdk
reply = await restored.chat_a("Could I order six?")
print(reply.response)


## Inspect the submitted history

With the default HTTP `store=False`, the next request contains the complete local transcript. Editing an earlier entry keeps the later recorded entries.


In [ ]:
assert "{sku}" in helper.yaml
assert requests[1]["input"][2]["id"] == "rs_1"
assert "summary: []" not in thread.yaml
assert requests[1]["input"][2]["summary"] == []
assert requests[1]["input"][3]["phase"] == "final_answer"
assert "previous_response_id" not in requests[1]
print(json.dumps(requests[1]["input"], indent=2))
await sdk.close()
workspace.cleanup()
